# 01 — Data Inventory: Structure, Quality & Station Catalog

**Goal**: Build a master `station_catalog.csv` from all 2024 `-site.csv` files and audit completeness across all three data sources (2017-2022 hourly, 2022-2023 5-min, 2024 5-min). This file is a dependency for every downstream notebook.

**Outputs**:
- `outputs/station_catalog.csv` — one row per station with all metadata
- `outputs/completeness_2024.csv` — station × month completeness matrix
- `outputs/completeness_2022_2023.csv` — station × month completeness matrix
- `outputs/completeness_2017_2022.csv` — station × month completeness matrix
- All figures saved to `outputs/figures/01_inventory/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT       = Path("/Users/kevinli/Desktop/Research_Gtech_gdot")
PATH_2024  = ROOT / "gdot_2025_data_total" / "CCS_5Min_2024"
PATH_2023  = ROOT / "gdot_2025_data_total" / "GDOT_CCS_2022-2023"
PATH_HIST  = ROOT / "gdot_2025_data_total" / "GDoT_UGA_Directional_Export_2017-2022" / "directional"
OUT_DIR    = ROOT / "outputs"
FIG_DIR    = OUT_DIR / "figures" / "01_inventory"

for d in [OUT_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Plot style ─────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
CMAP_MISS = sns.color_palette(["#2ecc71", "#e74c3c", "#f39c12"])  # green/red/orange

print("Paths OK. Output dirs created.")
print(f"  2024 data : {PATH_2024}")
print(f"  2022-2023 : {PATH_2023}")
print(f"  2017-2022 : {PATH_HIST}")

## 1. Build Station Catalog from 2024 `-site.csv` files

Each station in the 2024 dataset ships with a `GDOT_CCS-<COSIT>-site.csv` that contains rich metadata. We pivot the Name/Value format into a wide table, one row per station.

In [ ]:
def load_site_csv(path: Path) -> dict:
    """Parse a Name/Value site CSV into a flat dict. Strip surrounding quotes."""
    df = pd.read_csv(path, dtype=str)
    df["Name"]  = df["Name"].str.strip().str.strip('"')
    df["Value"] = df["Value"].str.strip().str.strip('"')
    return dict(zip(df["Name"], df["Value"]))

site_files = sorted(PATH_2024.glob("*-site.csv"))
print(f"Found {len(site_files)} site files in 2024 dataset")

records = []
for f in site_files:
    cosit = f.stem.replace("GDOT_CCS-", "").replace("-site", "")
    row   = load_site_csv(f)
    row["cosit_raw"] = cosit          # original 12-digit cosit string
    records.append(row)

catalog = pd.DataFrame(records)

# ── Tidy column names ──────────────────────────────────────────────────────────
catalog.rename(columns={
    "Id":                          "uuid",
    "Cosit":                       "cosit",
    "Name":                        "station_name",
    "Description":                 "description",
    "Notes":                       "notes",
    "Latitude":                    "latitude",
    "Longitude":                   "longitude",
    "County":                      "county",
    "City":                        "city",
    "SpeedLimit":                  "speed_limit",
    "Functional Class":            "functional_class",
    "Functional Class+Urban/Rural":"fc_urban_rural",
    "Legacy FFC Code":             "legacy_ffc",
    "Urban Area Type":             "urban_area_type",
    "TMAS Station Id":             "tmas_station_id",
    "HPMS Facility Type":          "hpms_facility_type",
    "Road Section Length":         "road_section_length_mi",
    "Start Mile Point":            "start_mile_point",
    "End Mile Point":              "end_mile_point",
    "Bearing":                     "bearing",
    "CollectionInterval":          "collection_interval_sec",
    "Through Lanes":               "through_lanes",
    "District (or Division etc.)": "district",
    "County Code (FIPS)":          "county_fips",
    "On National Highway System":  "on_nhs",
    "Road Surface":                "road_surface",
    "Posted Route Number (TMAS)":  "tmas_route",
    "LRS Route Id":                "lrs_route_id",
    "Data Update Frequency":       "data_update_freq",
    "Directionality":              "directionality",
    "Protocol":                    "protocol",
}, inplace=True)

# ── Type conversions ───────────────────────────────────────────────────────────
catalog["latitude"]              = pd.to_numeric(catalog["latitude"],              errors="coerce")
catalog["longitude"]             = pd.to_numeric(catalog["longitude"],             errors="coerce")
catalog["road_section_length_mi"]= pd.to_numeric(catalog["road_section_length_mi"],errors="coerce")
catalog["through_lanes"]         = pd.to_numeric(catalog["through_lanes"],         errors="coerce")
catalog["collection_interval_sec"]= pd.to_numeric(catalog["collection_interval_sec"],errors="coerce")
catalog["bearing"]               = pd.to_numeric(catalog["bearing"],               errors="coerce")

# Derive district from station_name (e.g. "001-0183" → district 1)
catalog["district_num"] = catalog["station_name"].str.extract(r"^(\d+)-").astype(float)

# Save
catalog.to_csv(OUT_DIR / "station_catalog.csv", index=False)
print(f"\nStation catalog: {len(catalog)} rows × {len(catalog.columns)} columns")
print(f"Saved → {OUT_DIR / 'station_catalog.csv'}")
catalog[["station_name","county","city","fc_urban_rural","speed_limit","through_lanes","latitude","longitude"]].head(8)

## 2. Station Catalog Profiling

Characterise the fleet: functional class, urban/rural, county, district, lanes, NHS membership.

In [ ]:
print("=== STATION SUMMARY ===")
print(f"Total stations (2024): {len(catalog)}")
print(f"Unique counties      : {catalog['county'].nunique()}")
print(f"Unique districts     : {catalog['district_num'].nunique()}")
print(f"\nUrban / Rural split:")
print(catalog["urban_area_type"].value_counts().rename(index={"R: Rural": "Rural", "U: Urban": "Urban"}))
print(f"\nFunctional Class breakdown:")
print(catalog["fc_urban_rural"].value_counts().head(15))
print(f"\nThrough Lanes distribution:")
print(catalog["through_lanes"].value_counts().sort_index())
print(f"\nOn NHS: {catalog['on_nhs'].value_counts().to_dict()}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Station Catalog Profiling — 2024 GDoT CCS Network", fontsize=14, fontweight="bold")

# ── 1. Functional Class (sorted) ───────────────────────────────────────────────
ax = axes[0, 0]
fc_counts = catalog["fc_urban_rural"].value_counts()
bars = ax.barh(fc_counts.index[::-1], fc_counts.values[::-1], color="#3498db")
ax.set_xlabel("Number of Stations")
ax.set_title("By Functional Class + Urban/Rural")
for bar, val in zip(bars, fc_counts.values[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va="center", fontsize=9)

# ── 2. Urban vs Rural ──────────────────────────────────────────────────────────
ax = axes[0, 1]
ur = catalog["urban_area_type"].value_counts()
colors = ["#27ae60" if "R" in i else "#e67e22" for i in ur.index]
wedges, texts, autotexts = ax.pie(ur.values, labels=ur.index, autopct="%1.1f%%",
                                   colors=colors, startangle=90)
ax.set_title("Urban vs Rural")

# ── 3. Stations per district ───────────────────────────────────────────────────
ax = axes[0, 2]
dist_counts = catalog["district_num"].value_counts().sort_index()
ax.bar(dist_counts.index.astype(int).astype(str), dist_counts.values, color="#9b59b6")
ax.set_xlabel("District Number")
ax.set_ylabel("Number of Stations")
ax.set_title("Stations per GDOT District")
for x, val in zip(range(len(dist_counts)), dist_counts.values):
    ax.text(x, val + 0.3, str(val), ha="center", fontsize=9)

# ── 4. Through Lanes distribution ─────────────────────────────────────────────
ax = axes[1, 0]
lanes = catalog["through_lanes"].dropna().astype(int).value_counts().sort_index()
ax.bar(lanes.index.astype(str), lanes.values, color="#e74c3c")
ax.set_xlabel("Number of Through Lanes")
ax.set_ylabel("Number of Stations")
ax.set_title("Through Lanes Distribution")
for x, val in enumerate(lanes.values):
    ax.text(x, val + 0.3, str(val), ha="center", fontsize=9)

# ── 5. Top 20 counties by station count ───────────────────────────────────────
ax = axes[1, 1]
county_counts = catalog["county"].value_counts().head(20)
ax.barh(county_counts.index[::-1], county_counts.values[::-1], color="#1abc9c")
ax.set_xlabel("Number of Stations")
ax.set_title("Top 20 Counties by Station Count")

# ── 6. NHS membership ─────────────────────────────────────────────────────────
ax = axes[1, 2]
nhs = catalog["on_nhs"].value_counts()
ax.bar(nhs.index, nhs.values, color=["#2980b9", "#bdc3c7"])
ax.set_ylabel("Number of Stations")
ax.set_title("On National Highway System")
for x, val in enumerate(nhs.values):
    ax.text(x, val + 0.5, str(val), ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
fig.savefig(FIG_DIR / "01_station_catalog_profiling.png", bbox_inches="tight")
plt.show()
print(f"Saved → {FIG_DIR / '01_station_catalog_profiling.png'}")

## 3. Completeness Audit — 2024 5-min Data

For each station, we compute how many 5-minute bins were actually recorded vs. the expected number (288 bins/day × days in month). We flag months with < 80% completeness as "incomplete".

In [ ]:
from calendar import monthrange

def expected_5min_bins(year: int, month: int) -> int:
    """Expected 5-min bins per direction per month = 288 bins/day × days."""
    return 288 * monthrange(year, month)[1]

# ── Scan all 2024 data files ───────────────────────────────────────────────────
data_files_2024 = sorted(PATH_2024.glob("*-data-*.csv"))
print(f"Found {len(data_files_2024)} data files in 2024")

records_2024 = []
for f in data_files_2024:
    cosit = f.stem.split("-data-")[0].replace("GDOT_CCS-", "")
    df = pd.read_csv(f, usecols=["Start Time", "Direction", "Count"],
                     parse_dates=["Start Time"], low_memory=False)
    df.rename(columns={"Start Time": "time", "Count": "volume"}, inplace=True)
    df = df.dropna(subset=["time"])
    df["year"]  = df["time"].dt.year
    df["month"] = df["time"].dt.month
    df["cosit"] = cosit

    # Observed bins per direction per (year, month)  — count unique timestamps
    grp = (df.groupby(["cosit", "year", "month", "Direction"])["time"]
             .nunique()
             .reset_index(name="observed_bins"))
    records_2024.append(grp)

comp_2024_raw = pd.concat(records_2024, ignore_index=True)

# Sum across directions for total observed bins per station-month
comp_2024 = (comp_2024_raw.groupby(["cosit", "year", "month"])["observed_bins"]
             .sum().reset_index())

# Expected bins (sum across 2 directions if bidirectional — we'll normalise later)
comp_2024["expected_bins"] = comp_2024.apply(
    lambda r: expected_5min_bins(int(r.year), int(r.month)), axis=1)

# Directions count varies by station, so normalise by actual directions observed
dir_count = (comp_2024_raw.groupby(["cosit", "year", "month"])["Direction"]
             .nunique().reset_index(name="n_directions"))
comp_2024 = comp_2024.merge(dir_count, on=["cosit","year","month"])
comp_2024["expected_bins"] *= comp_2024["n_directions"]

comp_2024["completeness_pct"] = (comp_2024["observed_bins"] / comp_2024["expected_bins"] * 100).clip(upper=100)
comp_2024["complete"] = comp_2024["completeness_pct"] >= 80

# Join station name for readability
name_map = catalog.set_index("cosit_raw")["station_name"].to_dict()
comp_2024["station_name"] = comp_2024["cosit"].map(name_map)

comp_2024.to_csv(OUT_DIR / "completeness_2024.csv", index=False)
print(f"\nCompleteness table: {len(comp_2024)} station-month records")
print(f"% complete months (≥80%): {comp_2024['complete'].mean()*100:.1f}%")
comp_2024.head(8)

In [ ]:
# ── Missing data audit: % missing per station (sorted worst → best) ────────────
station_summary_2024 = (comp_2024.groupby("station_name")
    .agg(
        months_observed    = ("completeness_pct", "count"),
        mean_completeness  = ("completeness_pct", "mean"),
        complete_months    = ("complete", "sum"),
        min_completeness   = ("completeness_pct", "min"),
    )
    .reset_index()
    .sort_values("mean_completeness")
)
station_summary_2024["pct_missing"] = 100 - station_summary_2024["mean_completeness"]

print(f"Stations with mean completeness < 50%: {(station_summary_2024['mean_completeness'] < 50).sum()}")
print(f"Stations with ALL months complete (≥80%): {(station_summary_2024['complete_months'] == station_summary_2024['months_observed']).sum()}")
print("\n--- Worst 10 stations ---")
display(station_summary_2024.head(10))
print("\n--- Best 10 stations ---")
display(station_summary_2024.tail(10))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("2024 Data Completeness Audit", fontsize=14, fontweight="bold")

# ── 1. Distribution of completeness scores ─────────────────────────────────────
ax = axes[0]
ax.hist(station_summary_2024["mean_completeness"], bins=30, color="#3498db", edgecolor="white")
ax.axvline(80, color="#e74c3c", linestyle="--", linewidth=1.5, label="80% threshold")
ax.set_xlabel("Mean Completeness % (across observed months)")
ax.set_ylabel("Number of Stations")
ax.set_title("Completeness Score Distribution")
ax.legend()

# ── 2. % missing by station — worst 40 ────────────────────────────────────────
ax = axes[1]
worst = station_summary_2024.head(40)
colors = ["#e74c3c" if m < 50 else "#f39c12" if m < 80 else "#27ae60"
          for m in worst["mean_completeness"]]
ax.barh(worst["station_name"], worst["pct_missing"], color=colors)
ax.set_xlabel("Mean % Missing")
ax.set_title("40 Worst Stations by Mean Missingness")
ax.axvline(20, color="#27ae60", linestyle="--", linewidth=1, alpha=0.7)
ax.axvline(50, color="#e74c3c", linestyle="--", linewidth=1, alpha=0.7)

# ── 3. Complete vs incomplete months per station (histogram) ──────────────────
ax = axes[2]
ax.hist(station_summary_2024["complete_months"], bins=range(0, 14), color="#9b59b6",
        edgecolor="white", align="left")
ax.set_xlabel("Number of Complete Months (≥80%)")
ax.set_ylabel("Number of Stations")
ax.set_title("Complete Months per Station (out of up to 12)")
ax.set_xticks(range(0, 13))

plt.tight_layout()
fig.savefig(FIG_DIR / "02_completeness_2024.png", bbox_inches="tight")
plt.show()
print(f"Saved → {FIG_DIR / '02_completeness_2024.png'}")

In [ ]:
# ── Station × Month completeness heatmap (2024) ────────────────────────────────
# Pivot: rows = stations (sorted by mean completeness), cols = year-month
pivot_2024 = comp_2024.copy()
pivot_2024["ym"] = pivot_2024["year"].astype(str) + "-" + pivot_2024["month"].astype(str).str.zfill(2)
heat = pivot_2024.pivot_table(index="station_name", columns="ym", values="completeness_pct")

# Sort rows: worst stations at top
row_order = station_summary_2024["station_name"].values  # already sorted worst→best
heat = heat.reindex(row_order)

fig, ax = plt.subplots(figsize=(14, max(8, len(heat) * 0.12)))
sns.heatmap(
    heat, ax=ax,
    cmap="RdYlGn", vmin=0, vmax=100,
    linewidths=0.3, linecolor="white",
    cbar_kws={"label": "Completeness %", "shrink": 0.5},
    xticklabels=True, yticklabels=(len(heat) <= 80)
)
ax.set_title("Station × Month Completeness Heatmap — 2024\n(sorted worst → best)", fontsize=13)
ax.set_xlabel("Year-Month")
ax.set_ylabel("Station")
plt.tight_layout()
fig.savefig(FIG_DIR / "03_completeness_heatmap_2024.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"Saved → {FIG_DIR / '03_completeness_heatmap_2024.png'}")

## 4. Completeness Audit — 2022-2023 5-min Data

The 2022-2023 data is stored as monthly CSVs across all stations, rather than per-station files. We load each file, identify unique stations, and compute the same completeness metric.

In [ ]:
monthly_files_2023 = sorted(PATH_2023.glob("*.csv"))
print(f"Found {len(monthly_files_2023)} monthly files in 2022-2023")

records_2023 = []
for f in monthly_files_2023:
    ym = f.stem  # e.g. "2022-01"
    year, month = int(ym.split("-")[0]), int(ym.split("-")[1])
    df = pd.read_csv(f, usecols=["Site", "Time", "Lane", "Volume"],
                     parse_dates=["Time"], low_memory=False)
    df["Site"] = df["Site"].str.strip().str.strip('"')
    df = df.dropna(subset=["Time"])

    # Count unique timestamps per site per lane-direction
    grp = (df.groupby(["Site", "Lane"])["Time"]
             .nunique().reset_index(name="observed_bins"))
    grp["year"]  = year
    grp["month"] = month
    records_2023.append(grp)

comp_2023_raw = pd.concat(records_2023, ignore_index=True)

# Sum across lanes for total observed bins per station-month
comp_2023 = (comp_2023_raw.groupby(["Site", "year", "month"])["observed_bins"]
             .sum().reset_index())

# Expected: 288 bins/day × days × n_lanes
lane_count = (comp_2023_raw.groupby(["Site", "year", "month"])["Lane"]
              .nunique().reset_index(name="n_lanes"))
comp_2023 = comp_2023.merge(lane_count, on=["Site", "year", "month"])
comp_2023["expected_bins"] = comp_2023.apply(
    lambda r: expected_5min_bins(int(r.year), int(r.month)) * r.n_lanes, axis=1)

comp_2023["completeness_pct"] = (comp_2023["observed_bins"] / comp_2023["expected_bins"] * 100).clip(upper=100)
comp_2023["complete"] = comp_2023["completeness_pct"] >= 80
comp_2023.rename(columns={"Site": "station_name"}, inplace=True)

comp_2023.to_csv(OUT_DIR / "completeness_2022_2023.csv", index=False)
print(f"\nCompleteness table: {len(comp_2023)} station-month records")
print(f"Unique stations    : {comp_2023['station_name'].nunique()}")
print(f"% complete months  : {comp_2023['complete'].mean()*100:.1f}%")
comp_2023.head(6)

## 5. Completeness Audit — 2017-2022 Hourly Data

Historical data is hourly (BinDuration=3600). Expected: 24 bins/day × days × 2 directions.

In [ ]:
hist_files = sorted(PATH_HIST.glob("*.csv"))
print(f"Found {len(hist_files)} monthly files in 2017-2022 hourly data")

records_hist = []
for f in hist_files:
    ym = f.stem  # e.g. "2017-08"
    year, month = int(ym.split("-")[0]), int(ym.split("-")[1])
    df = pd.read_csv(f, usecols=["Site", "Time", "Direction", "Volume"],
                     parse_dates=["Time"], low_memory=False)
    df["Site"] = df["Site"].str.strip().str.strip('"')
    df = df.dropna(subset=["Time"])

    grp = (df.groupby(["Site", "Direction"])["Time"]
             .nunique().reset_index(name="observed_bins"))
    grp["year"]  = year
    grp["month"] = month
    records_hist.append(grp)

comp_hist_raw = pd.concat(records_hist, ignore_index=True)

comp_hist = (comp_hist_raw.groupby(["Site", "year", "month"])["observed_bins"]
             .sum().reset_index())

dir_count_h = (comp_hist_raw.groupby(["Site", "year", "month"])["Direction"]
               .nunique().reset_index(name="n_directions"))
comp_hist = comp_hist.merge(dir_count_h, on=["Site","year","month"])

# Hourly: expected = 24 × days × directions
comp_hist["expected_bins"] = comp_hist.apply(
    lambda r: 24 * monthrange(int(r.year), int(r.month))[1] * r.n_directions, axis=1)

comp_hist["completeness_pct"] = (comp_hist["observed_bins"] / comp_hist["expected_bins"] * 100).clip(upper=100)
comp_hist["complete"] = comp_hist["completeness_pct"] >= 80
comp_hist.rename(columns={"Site": "station_name"}, inplace=True)

comp_hist.to_csv(OUT_DIR / "completeness_2017_2022.csv", index=False)
print(f"\nCompleteness table: {len(comp_hist)} station-month records")
print(f"Unique stations    : {comp_hist['station_name'].nunique()}")
print(f"% complete months  : {comp_hist['complete'].mean()*100:.1f}%")
comp_hist.head(6)

## 6. Cross-Source Comparison

Which stations appear in all three data sources? How does station coverage change across the 2017-2024 span?

In [ ]:
stations_2024 = set(comp_2024["station_name"].dropna().unique())
stations_2023 = set(comp_2023["station_name"].dropna().unique())
stations_hist = set(comp_hist["station_name"].dropna().unique())

in_all_three  = stations_2024 & stations_2023 & stations_hist
in_2023_2024  = stations_2024 & stations_2023
in_hist_2024  = stations_2024 & stations_hist
only_2024     = stations_2024 - stations_2023 - stations_hist

print("=== STATION COVERAGE ACROSS SOURCES ===")
print(f"  Stations in 2024 data            : {len(stations_2024)}")
print(f"  Stations in 2022-2023 data       : {len(stations_2023)}")
print(f"  Stations in 2017-2022 hourly     : {len(stations_hist)}")
print(f"  In all three sources (full span) : {len(in_all_three)}")
print(f"  In 2022-2023 AND 2024            : {len(in_2023_2024)}")
print(f"  In 2017-2022 AND 2024            : {len(in_hist_2024)}")
print(f"  Only in 2024 (new stations)      : {len(only_2024)}")

# Station-year presence table
from itertools import product

all_stations = stations_2024 | stations_2023 | stations_hist
presence = []
for st in sorted(all_stations):
    row = {
        "station": st,
        "in_hist_2017_2022": st in stations_hist,
        "in_5min_2022_2023": st in stations_2023,
        "in_5min_2024":      st in stations_2024,
    }
    row["n_sources"] = sum([row["in_hist_2017_2022"], row["in_5min_2022_2023"], row["in_5min_2024"]])
    presence.append(row)

presence_df = pd.DataFrame(presence)
presence_df.to_csv(OUT_DIR / "station_source_presence.csv", index=False)
print(f"\nSaved station_source_presence.csv ({len(presence_df)} rows)")

# Venn-style summary
print("\n=== SOURCE COMBINATION COUNTS ===")
print(presence_df.groupby(["in_hist_2017_2022","in_5min_2022_2023","in_5min_2024"]).size().rename("count").reset_index().to_string(index=False))

In [ ]:
# ── Stations active per year — timeline chart ──────────────────────────────────
# Build year-level presence from all three completeness tables
def to_year_station(comp_df, name_col="station_name"):
    return comp_df[[name_col, "year"]].drop_duplicates().rename(columns={name_col: "station"})

ys_hist = to_year_station(comp_hist)
ys_2023 = to_year_station(comp_2023)
ys_2024 = to_year_station(comp_2024)

all_ys = pd.concat([ys_hist, ys_2023, ys_2024]).drop_duplicates()
stations_per_year = all_ys.groupby("year")["station"].nunique().sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(stations_per_year.index.astype(str), stations_per_year.values, color="#2980b9")
ax.set_xlabel("Year")
ax.set_ylabel("Number of Active Stations")
ax.set_title("Number of Stations with Any Data, by Year (2017–2024)")
for x, (yr, cnt) in enumerate(stations_per_year.items()):
    ax.text(x, cnt + 1, str(cnt), ha="center", fontsize=10)
plt.tight_layout()
fig.savefig(FIG_DIR / "04_stations_per_year.png", bbox_inches="tight")
plt.show()
print(f"Saved → {FIG_DIR / '04_stations_per_year.png'}")

## 7. Completeness by Functional Class & Urban/Rural

Does missingness correlate with road type? Rural minor arterials with older sensors may be worse than urban interstates.

In [ ]:
# Join catalog metadata onto 2024 completeness
meta_cols = ["station_name", "fc_urban_rural", "urban_area_type", "functional_class",
             "county", "district_num", "through_lanes", "on_nhs"]
comp_2024_meta = comp_2024.merge(
    catalog[meta_cols], on="station_name", how="left"
)

# Mean completeness by functional class
fc_comp = (comp_2024_meta.groupby("fc_urban_rural")["completeness_pct"]
           .agg(["mean","median","count"])
           .sort_values("mean", ascending=False)
           .reset_index())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Data Completeness by Road Classification (2024)", fontsize=13, fontweight="bold")

# ── Box plot: completeness by functional class ──────────────────────────────────
ax = axes[0]
order = fc_comp["fc_urban_rural"].tolist()
data_by_fc = [comp_2024_meta[comp_2024_meta["fc_urban_rural"] == fc]["completeness_pct"].dropna()
              for fc in order]
bp = ax.boxplot(data_by_fc, vert=False, patch_artist=True,
                medianprops=dict(color="black", linewidth=2))
colors_fc = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(order)))
for patch, color in zip(bp["boxes"], colors_fc):
    patch.set_facecolor(color)
ax.set_yticklabels(order, fontsize=9)
ax.set_xlabel("Completeness %")
ax.set_title("Completeness Distribution by Functional Class")
ax.axvline(80, color="#e74c3c", linestyle="--", linewidth=1.2, alpha=0.7)

# ── Urban vs Rural completeness distribution ───────────────────────────────────
ax = axes[1]
for label, grp in comp_2024_meta.groupby("urban_area_type"):
    ax.hist(grp["completeness_pct"].dropna(), bins=30, alpha=0.6, label=label, density=True)
ax.axvline(80, color="#e74c3c", linestyle="--", linewidth=1.5, label="80% threshold")
ax.set_xlabel("Completeness %")
ax.set_ylabel("Density")
ax.set_title("Urban vs Rural Completeness Distribution")
ax.legend()

plt.tight_layout()
fig.savefig(FIG_DIR / "05_completeness_by_road_class.png", bbox_inches="tight")
plt.show()

# Summary table
print("=== MEAN COMPLETENESS BY FUNCTIONAL CLASS (2024) ===")
print(fc_comp.to_string(index=False))
print(f"\nSaved → {FIG_DIR / '05_completeness_by_road_class.png'}")

## 8. Event Flag Overview (2024)

Each station's `-event.csv` logs quality flags (Volume Bad, Class Bad, Holiday, Accident, Roadworks, etc.). Here we aggregate event type frequency across the entire network.

In [ ]:
event_files = sorted(PATH_2024.glob("*-event.csv"))
print(f"Found {len(event_files)} event files")

event_dfs = []
for f in event_files:
    cosit = f.stem.split("-event")[0].replace("GDOT_CCS-", "")
    df = pd.read_csv(f, parse_dates=["Start Time", "End Time"], low_memory=False)
    df["cosit"] = cosit
    event_dfs.append(df)

events = pd.concat(event_dfs, ignore_index=True)
events.rename(columns={"Start Time": "start", "End Time": "end",
                        "Event Type": "event_type_id",
                        "Flag": "flag", "Description": "description"}, inplace=True)
events["duration_hours"] = (events["end"] - events["start"]).dt.total_seconds() / 3600
events["station_name"]   = events["cosit"].map(name_map)

print(f"\nTotal event records : {len(events):,}")
print(f"Unique stations with events: {events['cosit'].nunique()}")
print(f"\nEvent flag frequency:")
flag_counts = events["flag"].value_counts()
print(flag_counts.to_string())

# Save
events.to_csv(OUT_DIR / "all_events_2024.csv", index=False)
print(f"\nSaved → {OUT_DIR / 'all_events_2024.csv'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Event Flag Overview — 2024 Network", fontsize=13, fontweight="bold")

# ── 1. Event type frequency (bar chart) ───────────────────────────────────────
ax = axes[0]
fc = flag_counts.sort_values(ascending=True)
bar_colors = ["#e74c3c" if "Bad" in i else
              "#f39c12" if "Atypical" in i or "Outlier" in i else
              "#3498db" if "Holiday" in i else
              "#9b59b6"
              for i in fc.index]
ax.barh(fc.index, fc.values, color=bar_colors)
ax.set_xlabel("Number of Event Records")
ax.set_title("Event Flag Frequency (all stations, 2024)")
for i, (flag, cnt) in enumerate(zip(fc.index, fc.values)):
    ax.text(cnt + max(fc.values)*0.005, i, f"{cnt:,}", va="center", fontsize=9)

# ── 2. Total flagged hours per station (top 30) ────────────────────────────────
ax = axes[1]
station_flag_hours = (events[events["flag"].str.contains("Bad|Atypical|Outlier", na=False)]
                      .groupby("station_name")["duration_hours"].sum()
                      .sort_values(ascending=False).head(30))
ax.barh(station_flag_hours.index[::-1], station_flag_hours.values[::-1], color="#e74c3c")
ax.set_xlabel("Total Flagged Hours (Bad/Atypical/Outlier)")
ax.set_title("Top 30 Stations by Total QC Flag Duration")

plt.tight_layout()
fig.savefig(FIG_DIR / "06_event_flag_overview.png", bbox_inches="tight")
plt.show()
print(f"Saved → {FIG_DIR / '06_event_flag_overview.png'}")

## 9. Summary Report

Print a consolidated summary of all key findings from this notebook. This is designed to be copy-pasteable into a report or slides.

In [ ]:
sep = "=" * 60

print(f"\n{sep}")
print("  GDoT DATA INVENTORY — SUMMARY REPORT")
print(f"{sep}")

print("\n[1] STATION CATALOG (2024)")
print(f"    Total stations           : {len(catalog)}")
print(f"    Unique counties          : {catalog['county'].nunique()}")
print(f"    Unique GDOT districts    : {int(catalog['district_num'].nunique())}")
rural_pct = (catalog["urban_area_type"].str.startswith("R").sum() / len(catalog) * 100)
print(f"    Rural stations           : {rural_pct:.1f}%")
print(f"    On NHS                   : {(catalog['on_nhs']=='Yes').sum()} stations")
print(f"    Most common FC           : {catalog['fc_urban_rural'].mode()[0]}")

print("\n[2] DATA COVERAGE ACROSS SOURCES")
print(f"    2017-2022 hourly         : {len(stations_hist)} stations")
print(f"    2022-2023 5-min          : {len(stations_2023)} stations")
print(f"    2024 5-min               : {len(stations_2024)} stations")
print(f"    In all 3 sources         : {len(in_all_three)} stations (full 2017-2024 span)")
print(f"    New in 2024 only         : {len(only_2024)} stations")

print("\n[3] COMPLETENESS (2024 5-MIN)")
c80 = (comp_2024["completeness_pct"] >= 80).mean() * 100
c50 = (comp_2024["completeness_pct"] < 50).mean() * 100
print(f"    Station-months ≥80%      : {c80:.1f}%  ← 'good' threshold")
print(f"    Station-months <50%      : {c50:.1f}%  ← severe missingness")
worst5 = station_summary_2024.head(5)["station_name"].tolist()
print(f"    5 worst stations         : {worst5}")

print("\n[4] COMPLETENESS (2022-2023 5-MIN)")
c80_23 = (comp_2023["completeness_pct"] >= 80).mean() * 100
print(f"    Station-months ≥80%      : {c80_23:.1f}%")

print("\n[5] COMPLETENESS (2017-2022 HOURLY)")
c80_h = (comp_hist["completeness_pct"] >= 80).mean() * 100
print(f"    Station-months ≥80%      : {c80_h:.1f}%")

print("\n[6] EVENT FLAGS (2024)")
top3_flags = flag_counts.head(3)
for flag, cnt in top3_flags.items():
    print(f"    {flag:<25}: {cnt:,} records")

print(f"\n{sep}")
print("  OUTPUT FILES")
print(f"{sep}")
for f in sorted(OUT_DIR.glob("*.csv")):
    print(f"    {f.name}")
for f in sorted(FIG_DIR.glob("*.png")):
    print(f"    figures/01_inventory/{f.name}")
print()